# Build a AI SQL agent

**What are we building?** A shop assistant. We ask a question such as *Which customer spent the most?* in plain English, and it looks up the answer in a database. At the end, we put the same Python code behind a Streamlit form.

**Our approach**

1. Create a shop database with customers and orders.
2. Ask an AI model to turn a question into SQL.
3. Run that SQL in SQLite and collect the matching rows.
4. Ask the model to explain those rows in plain English.

`Question → SQL → Database rows → Answer`

**Tech stack:** Python runs the program; SQLite stores the sample data in `shop.db`; LangChain connects prompts to the OpenAI model; Streamlit gives us a simple web UI; Jupyter lets us learn each step interactively.


## Before we start

Run this notebook from the project folder. Install `requirements.txt`, copy `.env.example` to `.env`, and put your OpenAI API key in `.env`. We load the key from that file so it does not appear in our code or video. Each question makes two model calls.


## 1. Import our tools

`sqlite3` is built into Python. `ChatPromptTemplate` prepares the instructions and question, `ChatOpenAI` calls the model, and `StrOutputParser` gives us plain text. `DB_PATH` names the local database file.


In [1]:
# %% 1. Imports and settings (same cell in the notebook)
import os
import sqlite3
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Read the API key from .env without putting it in the code.
load_dotenv()
DB_PATH = Path("shop.db")


## 2. Where does `shop.db` come from?

We do **not** download a database. The next function creates `shop.db` automatically when it connects to SQLite. It creates two tables:

| Table | What it stores |
| --- | --- |
| `customers` | Customer name and city |
| `orders` | Product, amount, and the customer's ID |

`orders.customer_id` links each order to `customers.id`. That link lets SQL join the tables to answer questions like *Who bought a monitor?* The sample inserts use `INSERT OR IGNORE`, so rerunning this cell does not create duplicate rows.


In [2]:
# %% 2. Create a small sample database (same cell in the notebook)
def setup_database():
    # Connecting creates shop.db if it does not exist yet.
    with sqlite3.connect(DB_PATH) as connection:
        # Create two tables: each order belongs to one customer.
        connection.executescript("""
            CREATE TABLE IF NOT EXISTS customers (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                city TEXT NOT NULL
            );
            CREATE TABLE IF NOT EXISTS orders (
                id INTEGER PRIMARY KEY,
                customer_id INTEGER NOT NULL,
                product TEXT NOT NULL,
                amount REAL NOT NULL,
                FOREIGN KEY (customer_id) REFERENCES customers(id)
            );
        """)
        # Add a few rows so we have something to ask questions about.
        # INSERT OR IGNORE avoids duplicates when this function runs again.
        connection.executemany(
            "INSERT OR IGNORE INTO customers VALUES (?, ?, ?)",
            [(1, "Asha", "Mumbai"), (2, "Ravi", "Delhi"),
             (3, "Meera", "Mumbai"), (4, "Kabir", "Pune")],
        )
        connection.executemany(
            "INSERT OR IGNORE INTO orders VALUES (?, ?, ?, ?)",
            [(1, 1, "Keyboard", 1200), (2, 1, "Mouse", 600),
             (3, 2, "Monitor", 9000), (4, 3, "Keyboard", 1200),
             (5, 3, "Monitor", 9000), (6, 4, "Mouse", 600)],
        )


Now call the function. After this cell runs, you will see `shop.db` in the project folder.


In [3]:
setup_database()


### Check that the database works

Before using AI, let's run a regular SQL query ourselves. `SELECT * FROM customers` means *show every column and every row* in the `customers` table. Each printed tuple is `(id, name, city)`. You should see four customers.


In [4]:
# Open shop.db and show all customers.
with sqlite3.connect(DB_PATH) as connection:
    for row in connection.execute("SELECT * FROM customers"):
        print(row)


(1, 'Asha', 'Mumbai')
(2, 'Ravi', 'Delhi')
(3, 'Meera', 'Mumbai')
(4, 'Kabir', 'Pune')


## 3. How does the question become an answer?

The code below splits the work into three small functions. `make_sql` gives the model the **schema** (table names, columns, and their relationship) and asks for one `SELECT` query. `get_rows` checks that response and runs the query. `sqlite3.Row` keeps column names with the values. `explain_rows` asks the model to explain the rows. Finally, `ask_database` calls these functions in order.

The model may write imperfect SQL, so we show the SQL and rows when we test it.


In [5]:
# %% 3. Turn a question into SQL, then answer it (same cell in the notebook)
def make_sql(question, model):
    # The schema tells the model which tables and columns it may use.
    schema = """customers(id INTEGER, name TEXT, city TEXT)
orders(id INTEGER, customer_id INTEGER, product TEXT, amount REAL)
orders.customer_id refers to customers.id. amount is in rupees."""

    # Ask the model to write one SQLite query for the question.
    sql_prompt = ChatPromptTemplate.from_messages([
        ("system", """You write SQLite queries for a beginner demo.
Use only this schema:\n{schema}
Return exactly one read-only SELECT query, with no markdown or explanation.
Use a JOIN when needed. Limit lists to 10 rows."""),
        ("human", "{question}"),
    ])
    sql = (sql_prompt | model | StrOutputParser()).invoke(
        {"schema": schema, "question": question}
    )
    return sql.strip().rstrip(";")


def get_rows(sql):
    # Reject responses that are not a single SELECT query.
    if not sql.upper().startswith("SELECT") or ";" in sql:
        raise ValueError("The model did not return one SELECT query.")

    # Open the database and fetch up to 10 matching rows.
    with sqlite3.connect(DB_PATH) as connection:
        # sqlite3.Row lets us use column names such as "name" and "amount".
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql).fetchmany(10)

        # Turn each SQLite row into a plain dictionary for display and the model.
        results = []
        for row in rows:
            results.append(dict(row))
        return results


def explain_rows(question, rows, model):
    # Ask the model to turn the returned rows into a short answer.
    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the question briefly using only these SQL rows: {rows}. "
                   "If there are no rows, say no matching data was found."),
        ("human", "{question}"),
    ])
    return (answer_prompt | model | StrOutputParser()).invoke(
        {"rows": str(rows), "question": question}
    )


def ask_database(question):
    # A single question follows the same three steps every time.
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError("Set OPENAI_API_KEY in a .env file before asking a question.")

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    sql = make_sql(question, model)
    rows = get_rows(sql)
    answer = explain_rows(question, rows, model)
    return sql, rows, answer


## 4. Try the first question

Look at all three outputs: the SQL the model wrote, the rows SQLite returned, and the final answer. Who has the largest total?


In [6]:
question = "Which customer spent the most?"
sql, rows, answer = ask_database(question)
print("SQL:", sql)
print("Rows:", rows)
print("Answer:", answer)


SQL: SELECT c.name, SUM(o.amount) AS total_spent 
FROM customers c 
JOIN orders o ON c.id = o.customer_id 
GROUP BY c.id 
ORDER BY total_spent DESC 
LIMIT 10
Rows: [{'name': 'Meera', 'total_spent': 10200.0}, {'name': 'Ravi', 'total_spent': 9000.0}, {'name': 'Asha', 'total_spent': 1800.0}, {'name': 'Kabir', 'total_spent': 600.0}]
Answer: Meera spent the most, with a total of 10200.0.


## 5. Try a question that needs a JOIN

Sales by city needs the city from `customers` and the amount from `orders`, so the SQL should join the two tables. Try changing the question afterward.


In [7]:
question = "What is the total sales amount by city?"
sql, rows, answer = ask_database(question)
print("SQL:", sql)
print("Rows:", rows)
print("Answer:", answer)


SQL: SELECT c.city, SUM(o.amount) AS total_sales
FROM customers c
JOIN orders o ON c.id = o.customer_id
GROUP BY c.city
LIMIT 10
Rows: [{'city': 'Delhi', 'total_sales': 9000.0}, {'city': 'Mumbai', 'total_sales': 12000.0}, {'city': 'Pune', 'total_sales': 600.0}]
Answer: - Delhi: 9000.0
- Mumbai: 12000.0
- Pune: 600.0


## 6. Put the same steps in Streamlit

Open `app.py`. It follows the same question → SQL → rows → answer flow. The final section adds a text box, an Ask button, and places to show the answer, SQL, and rows. Run `streamlit run app.py` from the project folder.

**Recap:** SQLite stores data, the first model call writes SQL, Python runs it, the second model call explains it, and Streamlit displays it.
